In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import anndata as ad
import scipy as sp
import pandas as pd
import pickle

from tqdm.notebook import tqdm
from scFM_density_estimation.models import *
from sklearn.model_selection import train_test_split

In [3]:
class ConditionalFlowMatchingWithScore(L.LightningModule):
    def __init__(self,
                 input_dim: int,
                 cond_dim: int,
                 hidden_dims: list = [],
                 cond_hidden_dims: list = [16],
                 cond_out_dim: int = 8,
                 lr: float = 1e-3,
                 use_encoder: bool = False,
                 use_ot_sampler: bool = False,
                 ot_method: str = "exact",
                 dropout: float = 0,
                 sigma_min: float = None,
                 ):
        super().__init__()
        self.save_hyperparameters()
        if not use_encoder:
            cond_out_dim = cond_dim
        
        self.cond_encoder = ConditionEncoder(cond_dim, cond_hidden_dims, cond_out_dim, dropout)
        self.mlp = FlowMatchingMLP(input_dim + 1 + cond_out_dim, hidden_dims, input_dim, dropout)
        self.ot_sampler = OTPlanSampler(method=ot_method)
        self.sigma_min = sigma_min
        
        self.use_encoder = use_encoder
        self.use_ot_sampler = use_ot_sampler
        self.cond_dim = cond_dim
        self.lr = lr

    def forward(self, x, t, cond):
        if t.dim() == 0 or t.size()[0] == 1:
            t = t.expand(x.shape[0]).unsqueeze(1)
        elif t.dim() == 1:
            t = t.unsqueeze(1)
            
        if self.use_encoder:
            cond_enc = self.cond_encoder(cond)
        else:
            cond_enc = cond
            
        xtc = torch.cat([x, t, cond_enc], dim=1)
        return self.mlp.mlp(xtc)

    def shared_step(self, x1, cond):
        device = x1.device
        
        x0 = torch.randn_like(x1).to(device)
        t = torch.rand(x1.shape[0]).unsqueeze(1).to(device)
        eps = torch.randn_like(x1).to(device)
        sigma_t = torch.sqrt(t * (1 - t))
        
        if self.use_ot_sampler:
            x0, x1, cond = self.sample_ot(x0, x1, cond)
        
        if self.sigma_min is None:
            xt = x0 + t * (x1 - x0)
            ut = (1 - 2 * t) / (2 * t * (1 - t) + 1e-8) * sigma_t * eps + x1 - x0
        else:
            xt = x0 + t * (x1 - (1 - self.sigma_min) * x0)
            ut = (1 - 2 * t) / (2 * t * (1 - t) + 1e-8) * sigma_t * eps + x1 - (1 - self.sigma_min) * x0

        pred_ut = self(xt + eps * sigma_t, t, cond)
        
        loss = F.mse_loss(pred_ut, ut)
        return loss
    
    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

def prepare_dataset(n, N, cond_dim, locs):
    C = np.random.randint(low=0, high=cond_dim, size=(N))
    X = np.concatenate([np.random.normal(loc=locs[c], scale=1, size=(1, n)) for c in C])

    X_train, X_test, C_train, C_test = train_test_split(X, C, test_size=10_000)

    X_train = torch.tensor(X_train).to("cuda").float()
    C_train = F.one_hot(torch.tensor(C_train).long(), num_classes=cond_dim).to("cuda").float()

    X_test = torch.tensor(X_test).to("cuda").float()
    C_test = F.one_hot(torch.tensor(C_test).long(), num_classes=cond_dim).to("cuda").float()
    return X_train, X_test, C_train, C_test

def initialize_model(n, cond_dim, sigma_min):
    model = ConditionalFlowMatchingWithScore(input_dim=n, hidden_dims=[1024, 1024, 1024], cond_dim=cond_dim,
                                    use_encoder=False, use_ot_sampler=False, sigma_min=sigma_min).to("cuda")
    optimizer = model.configure_optimizers()
    return model, optimizer

def train(batch_size, n_steps, model, optimizer, X, C):
    for k in range(n_steps):
        optimizer.zero_grad()
    
        indices = np.random.choice(range(X.shape[0]), size=batch_size, replace=False)
        loss = model.shared_step(X[indices], C[indices])
        
        loss.backward()
        optimizer.step()
    return model

def div_fn_hutch_trace(u):
    def div_fn(x, eps):
        _, vjpfunc = torch.func.vjp(u, x)
        return (vjpfunc(eps)[0] * eps).sum()

    return div_fn

class NODEWrapper_with_trace_div(torch.nn.Module):
    def __init__(self, model, cond):
        super().__init__()
        self.model = model
        self.cond = cond
        self.div_fn, self.eps_fn = div_fn_hutch_trace, torch.randn_like

    def forward(self, t, x, *args, **kwargs):
        x = x[..., :-1]
        
        def vecfield(y): 
            return self.model(y.unsqueeze(0), t, self.cond[:1]).squeeze()

        div = torch.vmap(self.div_fn(vecfield))(x, self.eps_fn(x))
        dx = self.model(x, t, self.cond)
        return torch.cat([dx, div[:, None]], dim=-1)

class NODEWrapper_with_ratio_tvf(torch.nn.Module):
    def __init__(self, model, control, condition, point):
        super().__init__()
        self.model = model
        self.cond_v = control
        self.cond_u = condition
        self.cond_f = point
        self.div_fn, self.eps_fn = div_fn_hutch_trace, torch.randn_like

    def forward(self, t, x, *args, **kwargs):
        x = x[..., :-1]
        
        def vecfield(y):
            return self.model(y.unsqueeze(0), t, self.cond_v[:1]).squeeze() \
            - self.model(y.unsqueeze(0), t, self.cond_u[:1]).squeeze()
        
        div = torch.vmap(self.div_fn(vecfield))(x, self.eps_fn(x))
        ut = self.model(x, t, self.cond_u)
        vt = self.model(x, t, self.cond_v)
        ft = self.model(x, t, self.cond_f)
        
        if model.sigma_min is None or model.sigma_min == 0:
            score_u = (t * ut - x) / (1 - t + 1e-2)
        else:
            score_u = (t * ut - x) / (1 - (1 - model.sigma_min) * t)
        correction_term_u = torch.linalg.vecdot(ft - ut, score_u)
        
        if model.sigma_min is None or model.sigma_min == 0:
            score_v = (t * vt - x) / (1 - t + 1e-2)
        else:
            score_v = (t * vt - x) / (1 - (1 - model.sigma_min) * t)
        correction_term_v = torch.linalg.vecdot(vt - ft, score_v)
        
        dr = div + correction_term_u + correction_term_v
        
        return torch.cat([ft, dr[:, None]], dim=-1)

def evaluate_model(model, data_samples, cond, condition, control, locs):
    device = data_samples.device

    log_condition_true = -0.5 * ((data_samples.cpu().numpy() - np.array(locs[np.argmax(condition)])) ** 2).sum(axis=1) - 0.5 * data_samples.shape[1] * np.log(2 * np.pi)
    log_control_true = -0.5 * ((data_samples.cpu().numpy() - np.array(locs[np.argmax(control)])) ** 2).sum(axis=1) - 0.5 * data_samples.shape[1] * np.log(2 * np.pi)
    log_ratio_true = log_condition_true - log_control_true

    # direct evaluation
    node = NeuralODE(
        NODEWrapper_with_trace_div(model, torch.tensor(condition).float().expand(data_samples.shape[0], cond_dim).to(device)),
        solver="dopri5", sensitivity="adjoint", atol=1e-4, rtol=1e-4
    )
    
    with torch.no_grad():
        traj = node.trajectory(
            torch.cat([data_samples, torch.zeros(data_samples.shape[0], 1).to(device)], dim=-1),
            t_span=torch.linspace(1, 0, 100).to(device)
        )
    z0, div = traj[-1][:, :-1], traj[-1][:, -1]
    log_condition_hat = (-0.5 * (z0 ** 2).sum(dim=1) - 0.5 * z0.shape[1] * np.log(2 * np.pi) + div).cpu().numpy()
    
    node = NeuralODE(
        NODEWrapper_with_trace_div(model, torch.tensor(control).float().expand(data_samples.shape[0], cond_dim).to(device)),
        solver="dopri5", sensitivity="adjoint", atol=1e-4, rtol=1e-4
    )
    
    with torch.no_grad():
        traj = node.trajectory(
            torch.cat([data_samples, torch.zeros(data_samples.shape[0], 1).to(device)], dim=-1),
            t_span=torch.linspace(1, 0, 100).to(device)
        )
    z0, div = traj[-1][:, :-1], traj[-1][:, -1]
    log_control_hat = (-0.5 * (z0 ** 2).sum(dim=1) - 0.5 * z0.shape[1] * np.log(2 * np.pi) + div).cpu().numpy()
    
    log_ratio_hat = log_condition_hat - log_control_hat

    # correction term - its own field
    node = NeuralODE(
        NODEWrapper_with_ratio_tvf(model, control=torch.tensor(control).float().expand(data_samples.shape[0], cond_dim).to(device),
                                   condition=torch.tensor(condition).float().expand(data_samples.shape[0], cond_dim).to(device), point=cond),
        solver="dopri5", sensitivity="adjoint", atol=1e-4, rtol=1e-4
    )
    
    with torch.no_grad():
        traj = node.trajectory(
            torch.cat([data_samples, torch.zeros(data_samples.shape[0], 1).to(device)], dim=-1),
            t_span=torch.linspace(1, 0, 100).to(device)
        )
    z0, ratio = traj[-1][:, :-1], traj[-1][:, -1]
    log_ratio_hat_v3 = -ratio.cpu().numpy()
    
    return log_ratio_true, log_ratio_hat, log_ratio_hat_v3

In [27]:
n = 10
N = 30_000
cond_dim = 4
batch_size = 512
n_steps = 50_000
n_tries = 10

condition = np.array([0, 1, 0, 0])
control = np.array([1, 0, 0, 0])

In [28]:
# easy
# locs = [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 2, 0, 2, 2, 2, 0, 0, 2], [2, 2, 0, 0, 2, 2, 2, 0, 0, 2], [-1, -1, 3, -1, 3, 3, 3, -1, -1, 3]]

# difficult
locs = [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [2, 2, 0, 0, 0, 0, 2, 2, 2, 2], [2, 0, 0, 2, 0, 0, 0, 2, 0, 0], [3, 3, 3, 3, -1, 3, 3, 3, 3, -1]]

In [29]:
sigmas = [1e-1, 1e-2, 1e-3, 1e-4, 0]
# names = ["true", "direct", "ct condition", "ct own", "ct control"]
names = ["true", "direct", "ct own"]
results = {sigma: [] for sigma in sigmas}

In [30]:
X_train, X_test, C_train, C_test = prepare_dataset(n, N, cond_dim, locs)

mask_condition = np.all(C_test.cpu().numpy() == condition, axis=1)
mask_control = np.all(C_test.cpu().numpy() == control, axis=1)
mask_both = mask_condition | mask_control

for sigma_min in sigmas:
    tmp_results = {name: np.array([]) for name in names}
    for _ in tqdm(range(n_tries)):
        model, optimizer = initialize_model(n, cond_dim, sigma_min)
        model = train(batch_size, n_steps, model, optimizer, X_train, C_train)
        log_ratios = evaluate_model(model, X_test, C_test, condition, control, locs)

        for i, name in enumerate(names):
            if tmp_results[name].shape[0] == 0:
                tmp_results[name] = log_ratios[i].reshape(-1, 1)
            else:
                tmp_results[name] = np.concatenate([tmp_results[name], log_ratios[i].reshape(-1, 1)], axis=1)

    results[sigma_min] = tmp_results

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/10 [00:00<?, ?it/s]

In [31]:
results["mask_condition"] = mask_condition
results["mask_control"] = mask_control
results["mask_both"] = mask_both

In [32]:
with open("./table_results/really_easy_results_lsm_3.pkl", "wb") as f:
    pickle.dump(results, f)

In [33]:
with open("./table_results/really_easy_results_lsm_3.pkl", "rb") as f:
    results = pickle.load(f)

mask_condition = results["mask_condition"]
mask_control = results["mask_control"]
mask_both = results["mask_both"]

In [34]:
def mean_confidence_interval(x, confidence=0.95):
    m, se = np.mean(x), sp.stats.sem(x)
    h = se * sp.stats.t.ppf((1 + confidence) / 2., x.shape[0] - 1)
    return np.round(m - h, 2), np.round(m, 2), np.round(m + h, 2)

In [35]:
processed_results = {e: {sigma: {name: np.array([]) for name in names} for sigma in sigmas} for e in ["sum", "mean"]}

for sigma in sigmas:
    for name in names:
        tmp_sum = np.sum(results[sigma][name][mask_condition], axis=0)
        tmp_mean = np.mean(results[sigma][name][mask_condition], axis=0)
        
        processed_results["sum"][sigma][name] = mean_confidence_interval(tmp_sum)
        processed_results["mean"][sigma][name] = mean_confidence_interval(tmp_mean)

In [36]:
df_mean = pd.DataFrame(processed_results["mean"]).transpose()
df_mean

,true,direct,ct own
0.1000,"(5.06, 5.06, 5.06)","(4.66, 5.11, 5.57)","(11.58, 12.56, 13.54)"
0.0100,"(5.06, 5.06, 5.06)","(4.91, 5.3, 5.69)","(18.9, 21.41, 23.93)"
0.0010,"(5.06, 5.06, 5.06)","(4.9, 5.54, 6.18)","(25.39, 29.19, 33.0)"
0.0001,"(5.06, 5.06, 5.06)","(4.53, 5.13, 5.73)","(37.07, 40.5, 43.93)"
0.0000,"(5.06, 5.06, 5.06)","(5.08, 5.38, 5.67)","(19.76, 21.18, 22.6)"


In [15]:
# difficult example
df_mean

,true,direct,ct own
0.1000,"(19.86, 19.86, 19.86)","(19.38, 21.07, 22.75)","(55.41, 59.29, 63.16)"
0.0100,"(19.86, 19.86, 19.86)","(20.85, 21.72, 22.59)","(104.44, 110.31, 116.18)"
0.0010,"(19.86, 19.86, 19.86)","(20.05, 21.9, 23.76)","(152.08, 166.99, 181.9)"
0.0001,"(19.86, 19.86, 19.86)","(18.52, 21.51, 24.49)","(185.6, 223.38, 261.15)"
0.0000,"(19.86, 19.86, 19.86)","(20.41, 22.98, 25.56)","(104.88, 112.35, 119.82)"


In [26]:
# "easy" example
df_mean

,true,direct,ct own
0.1000,"(20.05, 20.05, 20.05)","(15.7, 16.84, 17.97)","(42.28, 45.06, 47.85)"
0.0100,"(20.05, 20.05, 20.05)","(15.68, 17.44, 19.21)","(69.03, 79.41, 89.8)"
0.0010,"(20.05, 20.05, 20.05)","(16.23, 17.18, 18.14)","(105.27, 115.54, 125.81)"
0.0001,"(20.05, 20.05, 20.05)","(16.68, 18.07, 19.45)","(132.05, 152.49, 172.94)"
0.0000,"(20.05, 20.05, 20.05)","(16.06, 16.84, 17.63)","(67.88, 72.52, 77.16)"


In [37]:
# really easy example
df_mean

,true,direct,ct own
0.1000,"(5.06, 5.06, 5.06)","(4.66, 5.11, 5.57)","(11.58, 12.56, 13.54)"
0.0100,"(5.06, 5.06, 5.06)","(4.91, 5.3, 5.69)","(18.9, 21.41, 23.93)"
0.0010,"(5.06, 5.06, 5.06)","(4.9, 5.54, 6.18)","(25.39, 29.19, 33.0)"
0.0001,"(5.06, 5.06, 5.06)","(4.53, 5.13, 5.73)","(37.07, 40.5, 43.93)"
0.0000,"(5.06, 5.06, 5.06)","(5.08, 5.38, 5.67)","(19.76, 21.18, 22.6)"
